# **<u>Building Your Own AI Chatbot with Streamlit and HKUST GenAI</u>**

**A hands-on guide for ISOM3400**

---

> **Note on this notebook:** This `.ipynb` file is a *reading guide* plus a few runnable check cells. The actual chatbot must be saved as a separate `app.py` file and run from the terminal with `streamlit run app.py`. Streamlit apps do not run inside notebook cells.

## **1. Introduction: What Is a Chatbot, and Why Does the LLM Change Everything?**

### *1.1 Two Generations of Chatbots*

Before large language models (LLMs), most chatbots were **rule-driven**. They matched user input against pre-written keywords or decision trees and returned a canned answer. They were fast and predictable, but they could only handle interactions someone had explicitly anticipated.

LLM-powered chatbots work differently. They do **not** retrieve an answer from a database. Instead, they generate a response **token by token**, where each token is predicted based on all the text in the current conversation. This is called **autoregressive generation**. It explains why the same question can produce different answers, and why the model can respond sensibly to phrasing nobody programmed.

**Key idea:** A chatbot is not "smart" because it memorized answers. It is smart because it learned the statistical patterns of language and can generate a reasonable continuation given the context.

### *1.2 Why Streamlit?*

Streamlit turns a plain Python script into an interactive web app in minutes — no HTML, CSS, or JavaScript required. For a chatbot, Streamlit gives you three essential tools:

- `st.chat_input` — the input box at the bottom of the page.
- `st.chat_message` — renders messages as chat bubbles with user/assistant styling.
- `st.session_state` — stores conversation history across page reruns.

That last point matters: Streamlit reruns your entire script on every interaction. Without `st.session_state`, your chatbot would forget the conversation after every message.

### *1.3 Prompt Engineering: Getting the AI to Write Your Chatbot*

#### *1.3.1 What Is Prompt Engineering?*

A **prompt** is the instruction you give an LLM. **Prompt engineering** is the practice of writing that instruction so the model produces useful, correct, and usable output on the first or second try. The model does not read your mind — it only knows what your prompt says (plus whatever general knowledge it learned during training). Vague prompts produce vague code; specific prompts produce specific code.

Four principles matter most for code-generation prompts:

1. **State the role and goal explicitly.** Tell the model *what it is* ("You are a Python developer familiar with Streamlit") and *what the deliverable is* ("a complete `app.py`").
2. **Specify the constraints.** Name the exact libraries, the API endpoint, the model name, the file structure, and what must *not* appear in the code (e.g., hardcoded keys).
3. **Describe the expected behavior.** Walk through the interaction: what the user sees, what happens on each message, how history is preserved.
4. **Ask for the output format you want.** "Return only the contents of `app.py` in a single code block" prevents the model from padding the answer with prose you have to delete.

A useful mental model: **treat the prompt like a specification you would hand to a junior developer.** If a competent developer could not build the thing from your prompt alone, the model probably cannot either.

#### *1.3.2 A Worked Example Prompt*

Here is a prompt you can paste into an AI assistant (ChatGPT, Claude, Gemini, or the HKUST GenAI chat interface) to generate a simple chatbot wired to the HKUST GenAI API. Notice what it does **not** do: it does not tell the AI which Streamlit functions to use. It describes the **behavior** you want and lets the model choose the implementation. That is the right division of labor — you know what the app should do, the AI knows the library.

```text
You are a Python developer wanting to build a simple Python chatbot using Streamlit.

Build a single file, app.py, for a simple multi-turn chatbot that talks to
the HKUST GenAI API (an OpenAI-compatible endpoint).

What the app should do:
- Show a chat interface where the user can type a message and see the reply.
- Remember the conversation, so follow-up questions make sense.
- Use the model "gpt-4o-mini" with streaming turned off.
- Read the API key from Streamlit secrets under the name "HKUST_API_KEY".
  Never write the key directly in the code.
- Connect to base_url="https://hkust.azure-api.net/hkust-genai/v1" and send
  the key as an "api-key" header.
- If the API call fails, show the error in the chat instead of crashing.

Output format:
- Return ONLY the complete contents of app.py in a single ```python code block.
- No explanation before or after the code.
```

Why this prompt works:

- **Role + goal** are stated in the first two lines.
- **Behavior, not methods.** It says "show a chat interface," "remember the conversation," and "show the error in the chat" — the AI decides that this means `st.chat_message`, `st.session_state`, and a `try/except`. Students do not need to know those names in advance.
- **Only the non-negotiables are pinned down:** the package, the model, `stream=False`, the secrets key name, the base URL, and the header. These are project-specific facts the AI cannot guess — everything else is left to the model.
- **Output format** is explicit, so you get copy-pasteable code instead of a tutorial.

#### *1.3.3 What the AI Returns — and How It Maps to Appendix A*

A good response to the prompt above should look almost identical to **Appendix A**. Compare the two by behavior:

| What the prompt asked for | How Appendix A implements it |
|---|---|
| A chat interface the user can type into | `st.chat_input(...)` plus the `st.chat_message` bubbles |
| Remembering the conversation | `st.session_state.messages`, appended on every turn |
| Using the model with streaming off | `model=MODEL`, `stream=False` in the API call |
| Key from secrets, never hardcoded | `API_KEY = st.secrets["HKUST_API_KEY"]` |
| HKUST base URL + `api-key` header | The `OpenAI(base_url=..., default_headers=...)` block |
| Showing errors instead of crashing | The `try:` / `except Exception as e:` block |

The AI may choose **different methods** to achieve the same behavior — for example, a different way of laying out the page, or a helper function around the API call. That is normal and instructive: what matters is whether the *behavior* matches the prompt. Use Appendix A as your reference implementation, not as the only acceptable answer.

#### *1.3.4 Iterating on the Prompt*

Real prompt engineering is iterative. If the first response is wrong, do not rewrite the whole prompt — **add one constraint that targets the specific problem**:

- If the model hardcoded the key → add: *"The key must only be read from `st.secrets`; never assign it directly."*
- If the chatbot forgot earlier messages → add: *"The conversation history must be sent with every request so follow-up questions work."*
- If it added files you did not ask for → add: *"Do not include any file other than app.py."*
- If it used streaming → add: *"Hard requirement: streaming must be off."*

This loop — **run, observe, add a targeted constraint, rerun** — is the core skill. The prompt in 1.3.2 is a strong starting point, not a final answer.

#### *1.3.5 Try It Yourself*

Before moving on, do this once:

1. Paste the prompt from 1.3.2 into an AI assistant.
2. Save the returned code as `app.py` in a scratch folder.
3. Open **Appendix A** next to it and note any differences.

You now have both a working `app.py` and a feel for how prompt engineering turns a vague request into a deployable program. The rest of this guide (Sections 2–4) uses Appendix A as the reference code and walks through the local setup and deployment steps.

## **2. What the Code Does (Section by Section)**

The complete `app.py` is provided separately in Appendix A. Here is what each part is doing and why.

### *2.1 Secure API Configuration*

```python
MODEL = "gpt-4o-mini"
API_KEY = st.secrets["HKUST_API_KEY"]
CUSTOM_HEADERS = {"api-key": API_KEY}

client = OpenAI(
    base_url="https://hkust.azure-api.net/hkust-genai/v1",
    api_key=API_KEY,
    default_headers=CUSTOM_HEADERS
)
```

- The API key is **not** written in the script. It is read from a local secrets file. This keeps it out of your code and out of GitHub.
- `base_url` points to the HKUST GenAI endpoint, so requests go through HKUST's service rather than OpenAI's public API.

### *2.2 Giving the Chatbot Memory*

```python
if "messages" not in st.session_state:
    st.session_state.messages = []
```

`st.session_state` is a per-user storage box that survives page reruns. Without it, the message list would reset to empty after every interaction.

### *2.3 Displaying the Conversation*

```python
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])
```

On every rerun, the saved history is redrawn. `st.chat_message` accepts `"user"` and `"assistant"` roles and styles them differently.

### *2.4 Handling Input and Calling the API*

```python
if prompt := st.chat_input("Ask me anything about the course..."):
    ...
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": m["role"], "content": m["content"]} for m in st.session_state.messages],
        stream=False
    )
```

- The `:=` operator assigns and checks the value in one step.
- When the user sends a message, the code appends it to history and then sends **the entire history** to the API. This is how multi-turn conversation works: the model does not remember previous turns on its own — you must resend the context every time.
- `stream=False` means the full response is returned at once. This is a limitation of the HKUST endpoint, not a Streamlit issue.

## **3. Step-by-Step Local Setup**

### *3.1 Prerequisites*

- Python installed (3.11+ recommended).
- An API key from the HKUST ITSC portal (subscribe to the GenAI service first).
- A GitHub account.
- A Streamlit Community Cloud account (sign in with GitHub at share.streamlit.io).

### *3.2 Create the Project Folder and Install Packages*

Open Terminal (Mac) or Command Prompt or Anaconda Prompt (Windows), activate your virtual environment, then create a project folder:

```bash
mkdir hkust-chatbot
cd hkust-chatbot
```

Once done, navigate to VS Code and open the folder in VS Code (Go to `File` -> `Open Folder` -> `hkust-chatbot`). Create `app.py` in this folder and paste the provided chatbot code (see Appendix A).

Next, create a file named `requirements.txt` in the same folder with the following contents:

```text
streamlit
openai
```

After saving `requirements.txt`, install the packages from that file by running the following command on Terminal (Mac) or Command Prompt or Anaconda Prompt (Windows):

```bash
pip install -r requirements.txt
```

Using `requirements.txt` means the same dependency list can later be reused by Streamlit Community Cloud during deployment.

### *3.3 Create the Secrets File*

**What is `secrets.toml` and why do we need it?**

A `secrets.toml` file is a private configuration file that stores sensitive values — such as API keys, passwords, or tokens — that your app needs to run but that must **never** appear in your source code. Streamlit has built-in support for this file: when your app starts, Streamlit automatically reads `.streamlit/secrets.toml` and exposes its contents through `st.secrets`, so your code can use them without hardcoding them.

This matters for three reasons:

1. **Security:** If you hardcode an API key in `app.py`, anyone who sees your code (on GitHub, in a screenshot, in a shared folder) can use your key and consume your quota. Keeping it in `secrets.toml` means the key never appears in the code itself.
2. **Git safety:** `secrets.toml` lives in a local file that you add to `.gitignore`, so it is never uploaded to GitHub. Your code can be public while your key stays private.
3. **Deployment compatibility:** Streamlit Community Cloud uses the same mechanism. When you deploy, you paste the contents of `secrets.toml` into the platform's Secrets field, and the code reads `st.secrets["HKUST_API_KEY"]` exactly the same way — no code changes needed between local and cloud.

**How to create the file**

Your code reads `st.secrets["HKUST_API_KEY"]`, so you need a file at `.streamlit/secrets.toml`.

Execute the below commands using Terminal/Command Prompt/Anaconda Prompt:

```bash
mkdir .streamlit
nano .streamlit/secrets.toml
```

Type the following line, replacing the placeholder with your real key:

```toml
HKUST_API_KEY = "paste-your-actual-key-here"
```

Save and exit: `Ctrl + O`, `Enter`, `Ctrl + X`.

### *3.4 Test Locally*

```bash
streamlit run app.py
```

A browser window should open. If the chatbot responds to messages, your local setup is correct. You are now ready to deploy.

## **4. Deploying to GitHub and Streamlit Community Cloud**

### *4.1 Create the Repository with a Python `.gitignore`*

A `.gitignore` file is a plain text file that tells Git (and GitHub) which files and folders should be excluded from version control. When you upload files to a GitHub repository, Git normally tracks every file it sees, but some files should never be uploaded: temporary files, system files, and most importantly, files that contain secrets like API keys. 

By listing a file or folder name in `.gitignore`, you instruct Git to pretend that file does not exist, so it never gets committed or pushed to GitHub. This is essential for our project because `secrets.toml` contains your HKUST API key. If it were uploaded, anyone who visits your public repository could see the key and use up your quota.

This is the first deployment step because the `.gitignore` must exist before any files are uploaded. Otherwise, `secrets.toml` could be exposed.

1. Go to `github.com` and sign in.
2. Click **+** (top right) → **New repository**.
3. Name it, e.g., `hkust-chatbot`. Keep it **Public** (required for free Streamlit Cloud deployment).
4. Find the **Add .gitignore** dropdown and select **Python**. This makes GitHub generate a standard Python `.gitignore` for you.
5. Click **Create repository**.

### *4.2 Add the Secrets Line to `.gitignore`*

1. In your new repository, click the `.gitignore` file in the file list.
2. Click the **pencil icon** (Edit this file) in the upper right.
3. Scroll to the bottom of the file, add a new line, and type:

```text
.streamlit/secrets.toml
```
***Note:** You may already see this line inside the `.gitignore` file. If this is the case, no need to add it again.

4. Scroll down and click **Commit changes**.

The `.gitignore` now tells GitHub to ignore your secrets file. Because you never upload `secrets.toml` (it stays only on your computer), your key is protected.

### *4.3 Upload Your Files Through the GitHub Website*

Now upload your two code files through the browser — no commands needed.

1. On your repository's main page, click **Add file** → **Upload files**.
2. Drag and drop `app.py` and `requirements.txt` from your project folder into the browser window.
3. Scroll down, enter a short message such as `Add chatbot files`, and click **Commit changes**.

Your repository now contains `.gitignore`, `app.py`, and `requirements.txt`. Confirm that `.streamlit/secrets.toml` does **not** appear anywhere in the file list — it should stay only on your computer.

### *4.4 Deploy on Streamlit Community Cloud*

1. Go to share.streamlit.io and sign in with GitHub.
2. Click **Create app**.
3. When asked "Do you already have an app?", choose **Yup, I have an app**.
4. Fill in:
   - **Repository:** `YOUR_USERNAME/YOUR_REPO_NAME`
   - **Branch:** `main`
   - **Main file path:** `app.py`
5. **Critical step:** Click **Advanced settings** before deploying. In the **Secrets** field, paste the exact contents of your local `secrets.toml`:

   ```toml
   HKUST_API_KEY = "paste-your-actual-key-here"
   ```

   Click **Save**.
6. Click **Deploy**.

The build usually takes 1–3 minutes. Once live, test the chatbot in the browser.

## **5. Troubleshooting**

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'openai'` | Old openai version or missing install | Run `pip install -r requirements.txt` again, or `pip install "openai>=1.0.0"` |
| `401 Access denied due to invalid subscription key` | Key not valid, not subscribed, or not pasted into Streamlit Secrets | Verify the key with the curl command from HKUST ITSC; re-paste into Advanced settings |
| Works locally, 401 in the cloud | Secret not added in Streamlit Cloud | Re-enter the secret in Advanced settings and redeploy |
| `ModuleNotFoundError` in the cloud only | `requirements.txt` missing from the repository | Upload `requirements.txt` via **Add file → Upload files** and redeploy |
| Changes not showing after upload | Cloud app not rebuilt | Click **Reboot** in the app's settings menu |
| `FileNotFoundError` for secrets | `.streamlit/secrets.toml` missing locally | Recreate the folder and file as in Section 3.3 |
| `.streamlit/secrets.toml` appears in the GitHub file list | `.gitignore` missing the secrets line, or the file was uploaded manually | Edit `.gitignore` on GitHub (Section 4.2) and delete `secrets.toml` from the repository if present |

## **6. Key Takeaways**

- A chatbot's "intelligence" comes from **autoregressive token prediction**, not from a lookup table.
- **Prompt engineering** turns a vague request into a deployable program: state the role and goal, pin down constraints, describe behavior, and specify the output format.
- Multi-turn conversation requires **resending the full history** with every API call.
- `st.session_state` is what makes Streamlit apps remember anything between reruns.
- **Never hardcode API keys.** Use `st.secrets` locally and Streamlit Cloud's Secrets field in deployment.
- `secrets.toml` separates secrets from code; `.gitignore` keeps that file out of GitHub. GitHub can generate the `.gitignore` for you when you create the repository.

## **Appendix A: The Complete `app.py`**

Save the following as a separate file named `app.py` in your project folder. Do **not** run it inside a notebook cell — run it from the terminal with `streamlit run app.py`.

In [ ]:
import streamlit as st
from openai import OpenAI

# ================= Config =================
MODEL = "gpt-4o-mini"
API_KEY = st.secrets["HKUST_API_KEY"]
CUSTOM_HEADERS = {"api-key": API_KEY}

client = OpenAI(
    base_url="https://hkust.azure-api.net/hkust-genai/v1",
    api_key=API_KEY,
    default_headers=CUSTOM_HEADERS
)

# ================= Page Setup =================
st.title("HKUST Course Chatbot")
st.caption("Powered by HKUST GenAI")

# ================= Session State Init =================
if "messages" not in st.session_state:
    st.session_state.messages = []

# ================= Display History =================
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])

# ================= Handle Input =================
if prompt := st.chat_input("Ask me anything about the course..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        message_placeholder = st.empty()
        full_response = ""
        
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": m["role"], "content": m["content"]}
                    for m in st.session_state.messages
                ],
                stream=False
            )
            full_response = response.choices[0].message.content
        except Exception as e:
            full_response = f"Error: {str(e)}"
        
        message_placeholder.write(full_response)
    
    st.session_state.messages.append({"role": "assistant", "content": full_response})